# Fitting your model to real velocity field data

Our goal in this Notebook-4 is to finally use what we have built in the previous notebooks to fit our velocity field model to actual MaNGA data.

Finally, we will look at evidence from this galaxy that indicates the presence of dark matter!

In [ ]:
# Imports and preliminaries
import numpy as np
import numpy.ma as ma
import pandas as pd
from matplotlib import pyplot as plt
%matplotlib inline
#%matplotlib notebook
from astropy.io import fits
import astropy

# Import the `least_squares` function
from scipy.optimize import least_squares

# For auto reload of modules that have changed
%load_ext autoreload
%autoreload 2

In [ ]:
# Import the velocity_field class
# We assume that mangaRotCurves.py is in the working, notebook directory
from mangaRotCurves import velocity_field as vf

# Grab velocity field data for your galaxy from Marvin

- Using the `plateifu` ID for your blue galaxy, go to your galaxy's Marvin webpage by inputting the `plateifu` ID number in the form field at the top of the [Marvin website](https://dr17.sdss.org/marvin/).  

Note: This notebook assumes that you are accessing data products from "Data Release 17" (DR17).  That means that you should make sure you are logged into the DR17 version of Marvin https://dr17.sdss.org/marvin/.

- Find the blue button that says "Download" and select "Default MAPS" from the drop-down menu.  

The MAPS file contains automatic measurements, including velocity fields (!), that are performed by an automated "Data Analysis Pipeline" on every MaNGA galaxy.

- Open the MAPS fits file, following the steps for opening fits data in Notebook-2 of this project.
- Access the HDU called `EMLINE_GVEL`

`EMLINE_GVEL` contains velocity fields measured from various emission lines.  Remember that the velocity fields we've worked on so far we derived from the H$\alpha$ emission line.

In [ ]:
# Load the MaNGA MAPS file
# myPlateifu = ...  # Fill this in
# maps = ... # Fill this in
# maps.info()

In [ ]:
velmaps = maps['EMLINE_GVEL']    # velocity maps
masks = maps['EMLINE_GVEL_MASK'] # bad pixel masks for the velocity maps
ivars = maps['EMLINE_GVEL_IVAR'] # inverse variance maps
velmaps.data.shape

Notice the shape of these arrays.  When loading from fits, they come in as `(c, x, y)`, where `c` represents a dimension over the various emission lines that the MaNGA analysis pipeline uses to derive velocity fields.

- You need to figure out which index on `c` corresponds to $H\alpha$.

To do that, you could either read the MaNGA DR17 DAP documentation online, or you can inspect the "header" of the `EMLINE_GVEL` HDU.  Headers are commonly used to provide background information and metadata about the data in a file.

In [ ]:
# Uncomment the line below to read the header for the velocity maps HDU
#   Look for information about the emission lines corresponding to each map channel
# velmaps.header

In [ ]:
# Define the indx channel value for H-alpha
# HaIndx = ... # You fill this in.  (Are you counting from zero?)

In [ ]:
# Pull out just the H-alpha related velocity maps
HaVelmap = velmaps.data[HaIndx,:,:]
HaMask = masks.data[HaIndx,:,:]
HaIvar = ivars.data[HaIndx,:,:]

In [ ]:
# Apply the bad-pixel masks
# maskedHaVelmap = ma.masked_array(HaVelmap, mask=HaMask)
# maskedHaIvar = ... # You fill this in

## Plot the downloaded velocity field: Does it make sense

- Look back at the previous notebooks for examples of how to plot this velocity field (using a red-blue color map)
- Does the velocity field in your plot match what you see for your galaxy on the Marvin webpage?  Does it look like your approximate velocity field from Notebook-2?

It had better look similar, so if it doesn't you'll need troubleshoot what's going wrong.

In [ ]:
# Plot the velocity field
# Your code here...

## Plot the error map

Every measurement is made with some uncertainty.  A good data set will provide estimates of the "errors" of any set of measurements it provides.  In this case, we need a map of errors on the provided H$\alapha$ velocity map.  Those errors will help us fit our model to the data (pixels with large velocity errors, e.g. near the noisy edges of the data cube, will be downweighted in the model fit as a result of their larger errors.

We can get the error map from the "inverse variance" map. What's that?  Well, the "variance" is simply the square of the error value, $\sigma$.  So "inverse variance" is ${\rm IVAR} = 1 / \sigma^2$

- Using the IVAR 2D array, solve for $\sigma$ and plot the "error map" (i.e., the map of velocity uncertainties at every pixel). 
- What are the units?  (Hint: same as the velocity field units)

In [ ]:
# Make a map of the errors on the velocity measurements (from the inverse variance IVAR array)

# First solve for the map of errors
# maskedHaVelmapErrors = ... # You fill this in

# Then plot...
#   Note, it's good practice not to use the red-blue color map in this plot 
#   because you're not plotting velocities (which may be positive or negative), 
#   but errors on the velocities, which should always be positive.

Take a look at your error map.  Does it look reasonable?

# Fit our 2D model to the velocity field

## First, set up guesses for some initial parameters

We need to provide starting guesses for the "position angle" (PA), the inclination, and rotation curve parameters ($V_{\rm max}$ and $h$).

At least for PA and inclination, we can use measurements in the DRPAll catalog.

First, we will pull out $b/a$ (`ba`), the ratio of the minor-axis to the major-axis of the galaxy's light profile, as measured on the image.

Then, we will pull out `phi` the position angle of the major axis as seen on the sky, measured in degrees from North to East (remembering that in astronomical images, North is up and East usually points to the left).

In [ ]:
# Load the DRPAll file
#   See HW-5 on exploring MaNGA galaxies
#   Your code here
# DRPAll = ...

In [ ]:
# Find the indx value of DRPAll that matches your galaxy, identified by its "platifu" ID number
#   Because of string storage issues, we may need to strip away a b' ' around the platifu string
#   BTW: How do we know that 'plateifu' was available in DRPAll?  We can look at DRPAll.columns
indx = DRPAll['plateifu'].str.strip("b'").str.strip("'") == myPlateifu

In [ ]:
# Pull out the b/a and position angle values for the galaxy
ba = DRPAll['nsa_elpetro_ba'][indx].item()
phi = DRPAll['nsa_elpetro_phi'][indx].item() # degrees, East-from-North

In [ ]:
# Estimate inclination from the b/a
#   You need to figure out how to do this
# inclination = ...


## Perform the model fitting

In [ ]:
# Set up the pixel coordinates and create a coordinate grid for x and y
pix = np.arange(len(HaVelmap[:,0])).astype(float)
x, y = np.meshgrid(pix[::-1], pix)  # In the model we want +x to go to left

xMasked = ma.masked_array(x, mask=HaMask)
yMasked = ma.masked_array(y, mask=HaMask)

In [ ]:
# Instantiate the velocity object
#   Note: To use the masked arrays in `least_squared()`, we apply the `compressed()` function which 
#   yields 1D arrays from 2D arrays, with masked pixels removed
VF = vf(xMasked.compressed(), yMasked.compressed(), 
        np.clip(maskedHaVelmap.compressed(), -200,200), 
        maskedHaVelmapErrors.compressed())

In [ ]:
# Set up the fit parameters
# First two parameters: galaxy center (x,y) in pixels
# Third parameter: PA (degrees)
# Fourth parameter: inclination (degrees)
# Fifth parameter: Vmax (km/s)
# Sixth parameter: Rotation scale (turnover radius) (pixels)
# Seventh parameter: bulk velocity from redshift (km/s), probably 0 km/s if using MAPS

# guess_par = ... # You fill this in

In [ ]:
# Bound the fit values
# Because the velocity-field fitting has many parameters, and
# because some of those parameters have natural limits (e.g.,
# 0 < inclination < 90), we impose lower and upper boundaries
# on the allowed parameter values.  +/- np.inf means there is
# no limit.
lower_bounds = np.array([-40, -40,   0,  5,   1, 0.1, -300])
upper_bounds = np.array([  0,   0, 360, 85, 500, 100,  300])

In [ ]:
# Experiment with fixing certain input parameters
fix_par = np.array([ True, True, True, True, False, False, True])
fix_par[:] = False  # This statements allows all parameters to be free
VF.fix_par(guess_par, fix_par)

In [ ]:
# Run the fit, get the errors in the parameters
# result = least_squares(VF.chi, guess_par, bounds=(lower_bounds, upper_bounds))
free_par = np.invert(fix_par)
result = least_squares(VF.chi, guess_par[free_par],
                       bounds=(lower_bounds[free_par],
                               upper_bounds[free_par]))
covar = np.linalg.inv(np.dot(result.jac.T,result.jac))
par_err = np.sqrt(np.diagonal(covar)) # This gives us the errors on fit parameters

In [ ]:
# For each parameter, print out a nicely formatted table of
#   'Parameter', Best Fit Value', 'Error on Best Fit Value'

### Does the model make sense?

First, we want to check that the model fit didn't veer off into some strange part of parameter space.  Second, we want to make sure the fit results are not too close to the input parameters... that we are actually fitting _something_.

#### Warning on parameter errors

The parameter errors can often be misleadingly small in least-squares fitting.  Don't trust them.  Typically a Bayesian approach and a careful, sometimes separate, treatment is required to derive good error estimates on fit parameters.

## Compare measured and modeled velocity fields

In [ ]:
# Copy the velocity field object, since we're going to re-instantiate it
from copy import deepcopy
VF2 = deepcopy(VF)

# Redefine the x and y as 2D arrays for plotting (we earlier used `compressed()` 1D versions)
VF2.x = xMasked
VF2.y = yMasked

# Output the modeled 2D velocity field using the best-fit parameters in `result.x`
model = VF2.sample(result.x)

In [ ]:
# Plot the H-alpha velocity field (data) next to the best-fit model
# Note: `plt.subplots` lets you plot multiple panels in the same figure

scaleLimits = [-120,120] # Set velocity plotting limits so that they are the same for data and model

fig, ax = plt.subplots(1, 2)
dataPlot = ax[0].imshow(maskedHaVelmap, vmin=scaleLimits[0], vmax = scaleLimits[1],
                        origin='lower', interpolation='nearest', cmap='RdBu_r')
modelPlot = ax[1].imshow(model, vmin=scaleLimits[0], vmax = scaleLimits[1],
                         origin='lower', interpolation='nearest', cmap='RdBu_r')
plt.colorbar(modelPlot, ax=ax[1], fraction=0.046, pad=0.04) # fraction/pad for geting color bar height to match imshow
plt.colorbar(dataPlot, ax=ax[0], fraction=0.046, pad=0.04)
plt.subplots_adjust(wspace=0.6) # increase separation between plots
plt.show()

If all has gone well, you should see that the model (right panel) does a pretty good job matching the observed H$\alpha$ velocity field (left).

## Plot the rotation curve implied by the best fit


In [ ]:
# Use the best-fitting result to get the disk-plane polar coordinates
r, theta = VF.polar_coordinates(result.x[:4])

In [ ]:
# Deproject the rotation velocities; i.e., use some algebra to invert the Vlos equation
# above to solve for Vrot given Vlos
vrot = (VF.v - result.x[6]) / np.cos(np.radians(theta)) / np.sin(np.radians(result.x[3]))

In [ ]:
# The deprojection near the minor axes (where theta is +/- 90) is very large, so
# let's only plot data near the major axis.  Let's select data within a +/- 30 deg
# wedge around the major axis in indx
wedge = 30
indxWedge = ((theta > -wedge) & (theta < wedge)) | (theta > 180-wedge) | (theta < -180+wedge)

In [ ]:
# Plot the result

# First plot the data (e.g., using plt.scatter() on r[indx] and vrot[indx])
# Your code here

# Then overplot the best-fitting model
#   You'll need to generate an array of radii input values
#   Then use VF.rotation_curve() to return the model velocity as a function of radius,
#        given the bestfit parameters in result.X

Again, the model rotation curve (in red) should thread through the data pretty well.  Similar to what we saw in Notebook-3 for an artificial velocity field.

# Well... Is there dark matter?

We're going to answer this question by testing whether the "gravitating" mass implied by the fitted rotation curve can be explained by the visible mass we see in images of this galaxy.  

## First, estimate the amount of visible flux from stars

For a handle on how much "visible matter" lives in your galaxy, we will use an estimate of the mass it contains in stars. We can safely ignore the mass of gas and dust because observations tell us those components are just a few, maybe 10, percent of the stellar mass in the vast majority of galaxies today.  

As we see below, we can get an estimate of the total stellar mass from the DRPAll parent catalog file, but we are also interested in how the mass in stars interior to some location increases with radius from the galaxy's center.  From this measure of enclosed mass as a function of radius, $M_*(r)$, we can estimate the "rotation curve," $v_*(r)$, that would be expected from just the mass in stars, alone.

In [ ]:
# Grab the reconstructed image of your galaxy
#   Aim for the z-band image 
#   The redder bands (like i-band or z-band) are better tracers of the total stellar mass
#   Bluer bands (like u-band or g-band) can be more biased by rare young stars and dust
#   On the other hand, you might find blemishes in one band that would make another band's image preferable.
# mangaData = ... # You fill this in (look at Notebook-1)
# mangaData.info()

# img = mangaData['ZIMG'].data

In [ ]:
# Create a bad-pixel mask for this image using the image itself.
#   By displaying the image (see below), we see that bad pixels often have values >> 5

badPixelThreshold = 5 # Manually set.  Bad pixels are above this value.  You may need to experiment with this.

badpixelMask = img > badPixelThreshold

# This image has an associated mask
maskedImg = ma.masked_array(img, mask=badpixelMask)

# Here's a way of "filling in" the bad pixels
#   We assume that the image is centered and roughly symmetric
filledImg = img.copy()
filledImg[badpixelMask] = img[:,::-1][badpixelMask]

In [ ]:
# Plot the image with the mask applied or the one with approx values filled in
#   We should see that bad pixels are (mostly) white, which means they will be ignored in what follows
plt.imshow(filledImg, origin='lower', vmin=0, vmax=2)
plt.colorbar()
plt.show()

## Go from image flux to stellar mass to velocity

We'll use Astropy tools to help us with constants and units.

What we need to do:
- Convert pixel size (or number of pixels) to distance in kpc 
- Convert image flux to stellar mass, in appropriate units
- Solve the equation below from Notebook-1 (introductory notebook) for velocity in km/s

Relating orbital velocity to interior mass:
$$ m_{\rm star} v^2 = \frac{G M_{\rm interior} m_{\rm star}}{r} $$


In [ ]:
from astropy import units as u
from astropy import constants as const

# Import a cosmological model based on observations published in 2018 from the Planck satellite
from astropy.cosmology import Planck18 as cosmo  

### Start linking sky coordinates to physical kpc (kiloparsecs) within a galaxy 

To begin, we need to compute the number of kpc per arcsec (an arcsec is 1/60th of 1/60th of a degree, as seen on the sky).  For reference, half of a degree is about angular width of your finger at arm's length, as well as the angular width of the moon (and the sun) in the sky.

The relation between a galaxy's angle-on-sky and the physical distance within that galaxy depends (of course) on the distance to the galaxy (represented by its redshift).  As we know from everyday experience, the same physical object appears smaller ("subtends" a smaller angle) the further away it is from us.  But, at very large distances, cosmological model also matter a lot.  In effect, the spacetime of the universe expands while the photons are traveling from the galaxy to us.  So the concept of distance becomes a little tricky.


In [ ]:
# Define the relationship between on-sky angle and physical distance within the galaxy
redshift = DRPAll['nsa_z'][indx].item()

# Notice the helpful use of units here
kpcPerArcsec = cosmo.kpc_proper_per_arcmin(redshift).to(u.kpc/u.arcsec)
kpcPerArcsec

In [ ]:
# Define a series of disk radii samples, within the galaxy

radii = np.linspace(1,35,25)
nradii = len(radii)

# Radii in kpc
radiikpc = kpcPerArcsec * radii * 0.5 * u.arcsec

# Radii in meters
radiim = radiikpc.to(u.m)

### Compute stellar mass as a function of radius

First confirm that we have a 2D on-sky map of the disk radius, thanks to geometry worked out by the `VF.polar_coordinates()` function.

Then, by scaling to the total stellar mass, we can estimate the stellar mass enclosed at any radius. We will make the assumption that the image flux is one-to-one proportional to stellar mass.  

In [ ]:
# Confirm that we have a 2D disk radius map by plotting it.
#   The plot should have the rough shape of the galaxy light profile and 
#   should measure the distnace in pixels from the center **in the disk frame** 
#   (so tilted on the sky)
r2, theta2 = VF2.polar_coordinates(result.x[:4])

plt.imshow(r2, origin='lower')
plt.colorbar()
plt.show()

In [ ]:
# The total stellar mass of the galaxy from the DRPAll catalog
#   We need to scale by a cosmological parameter called little-h
totalMstar = DRPAll['nsa_elpetro_mass'][indx]/(cosmo.h**2) # in solar masses

In [ ]:
# Cumulative flux as a function of disk radii
cumFlux = np.zeros(nradii)
for i,rad in enumerate(radii):
    imgIndx = r2 < rad
#     cumFlux[i] = maskedImg[imgIndx].sum()
    cumFlux[i] = filledImg[imgIndx].sum()
    
# To convert to a cumulative stellar mass as a function fo disk radii
#   scale the total stellar mass by the relative flux fraction
#   Here we assume the stellar mass is directly proportional to flux
cumMstar = totalMstar.item() * cumFlux / cumFlux.max()

Solve this equation for $v$:
$$ m_{\rm star} v^2 = \frac{G M_{\rm interior} m_{\rm star}}{r} $$


In [ ]:
# Your solution here
#  You will benefit from `const.G` and `const.M_sun`


### Re-plot the rotation curve and data with the stars-only curve overplotted

In [ ]:
# Plot data and rotation curves
#   Your plot should include:
#     - Data points
#     - Best-fit model
#     - Expectation from stellar mass only
#   Try to plot your radii in kpc!
#
# Your code here

Were you able to get something like this?

![]()
<div>
<img src="example_rotCurve_wStars.jpg" width="400"/>
</div>


# Interpreting your rotation curve plot

- Overall, and thanks to observations first made by [Vera Rubin](https://en.wikipedia.org/wiki/Vera_Rubin) and confirmed now for many decades, we expect to see that your stars-only rotation curves falls short of the measured data and modeled rotation curve.  

If you see that the stars-only curve doesn't reach the data and model, especially at large radii, there must be something missing! ... **CONGRATULATIONS: you have successfully found strong evidence of dark matter!**, at least in this galaxy.

- You might find that your first few inner-most stars curve looks funny.  The spatial resolution of the data actually doesn't allow us to accurately measure what's happening below about 4 pixels or typically 1-2 kpc.

Another important insight:

- You will likely find that your rotation curve fit continues to rise (gently, perhaps) at large distances, while the expected stellar rotation curve starts to decrease.  

This is another kind of evidence for dark matter from rotation curves.  Dark matter is far more extended that luminous matter, so it continues to add gravitating mass as we consider larger and larger radii.  The luminous matter (stars) are much more confined.  So, as we consider radii further from the stellar distribution, the mass in stars no longer increases, but of course the radius does.  Eventually (according to the equation above), the velocity from stars-only will turn-over and decrease.

So, even if you struggled to get the units and overal normalization to match up when comparing stars-only to the observed/modeled rotation curve, the difference in the _shape_ of the curves tells us there must be unseen matter at large radii!